# Lab 5: Deep Learning & LLMs for NLP

**Course:** Natural Language Processing


**Objectives:**
- Understand RNN, LSTM, GRU architectures for sequence modeling
- Use pre-trained Transformers for NER
- Interact with LLMs via API for text generation

---

## Instructions

1. Complete all exercises marked with `# YOUR CODE HERE`
2. **Answer all written questions** in the designated markdown cells
3. Save your completed notebook
4. **Push to your Git repository and send the link to: yoroba93@gmail.com**

---

## Lab Structure

| Part | Model | Task |
|------|-------|------|
| A | RNN | Character-level Language Model |
| B | LSTM | Sentiment Analysis |
| C | GRU | News Classification |
| D | Transformer | Named Entity Recognition |
| E | LLM (Mistral) | Text Generation & QA |

---

## Setup

In [2]:
!pip install -U datasets huggingface_hub transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.33.2
    Uninstalling huggingface-hub-0.33.2:
      Successfully uninstalled huggingface-hub-0.33.2
  Attempting uninstall: datasets
    Found existing installation: datasets 3.6.0
    Uninstalling datasets-3.6.0:
      Successfully uninstalled datasets-3.6.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transformers

In [1]:
import datasets
import huggingface_hub
import transformers

print(datasets.__version__)
print(huggingface_hub.__version__)
print(transformers.__version__)

5.0.0
1.21.0
5.12.1


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.11.0+cpu


---

# PART A: RNN - Character-Level Language Model (10 min)

**Use Case:** Predict the next character for autocomplete.

**Dataset:** Tiny Shakespeare

In [5]:
# Load Tiny Shakespeare dataset
import requests

print("Downloading Tiny Shakespeare dataset...")

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text[:10000]   # First 10,000 characters

print(f"Loaded {len(text)} characters")
print("\nSample text:")
print(text[:500])

print(f"Text length: {len(text)} characters")
print(f"Sample: {text[:200]}")

Loaded 10000 characters

Sample text:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor
Text length: 10000 characters
Sample: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [6]:
# Create character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars[:30])}...")

Vocabulary size: 57
Characters: 
 !',-.:;?ABCDEFHIJLMNOPRSTUVW...


In [7]:
# Prepare sequences
seq_length = 30
X, y = [], []

for i in range(len(text) - seq_length):
    X.append([char_to_idx[c] for c in text[i:i+seq_length]])
    y.append(char_to_idx[text[i+seq_length]])

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print(f"Sequences: {X.shape[0]}, Sequence length: {seq_length}")

Sequences: 9970, Sequence length: 30


In [8]:
# Simple RNN model
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])  # Last timestep
        return out

# Create model
rnn_model = CharRNN(vocab_size, embed_dim=32, hidden_dim=64).to(device)
print(f"RNN Parameters: {sum(p.numel() for p in rnn_model.parameters()):,}")

RNN Parameters: 11,801


### Exercise A.1: Train the RNN

In [10]:
# Hyperparameters
batch_size = 128
epochs = 5
learning_rate = 0.001  # Chosen from the suggested range (0.001-0.01)

# DataLoader
dataset = TensorDataset(X[:5000], y[:5000])  # Use subset for speed
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(rnn_model.parameters(), lr=learning_rate)

# Training loop
losses = []

rnn_model.train()

for epoch in range(epochs):
    total_loss = 0

    for batch_X, batch_y in loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        # 1. Zero gradients
        optimizer.zero_grad()

        # 2. Forward pass
        outputs = rnn_model(batch_X)

        # 3. Compute loss
        loss = criterion(outputs, batch_y)

        # 4. Backward pass
        loss.backward()

        # 5. Update weights
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    losses.append(avg_loss)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/5, Loss: 3.5727
Epoch 2/5, Loss: 3.0045
Epoch 3/5, Loss: 2.8219
Epoch 4/5, Loss: 2.6821
Epoch 5/5, Loss: 2.5518


In [11]:
# Generate text
def generate_text(model, start_str, length=100):
    model.eval()
    chars_generated = list(start_str)
    input_seq = [char_to_idx.get(c, 0) for c in start_str[-seq_length:]]

    with torch.no_grad():
        for _ in range(length):
            x = torch.tensor([input_seq[-seq_length:]]).to(device)
            output = model(x)
            pred_idx = torch.argmax(output, dim=1).item()
            chars_generated.append(idx_to_char[pred_idx])
            input_seq.append(pred_idx)

    return ''.join(chars_generated)

# Test generation
print("Generated text:")
print(generate_text(rnn_model, "To be or not", length=100))

Generated text:
To be or not the the the the the the the the the the the the the the the the the the the the the the the the the


---

# PART B: LSTM - Sentiment Analysis

**Use Case:** Classify movie review sentiment.

**Dataset:** IMDB Reviews

In [7]:
from sklearn.datasets import fetch_20newsgroups

# Training data
train_data = fetch_20newsgroups(
    subset="train",
    categories=["rec.autos", "sci.med"],
    remove=("headers", "footers", "quotes"),
)

train_texts = train_data.data[:1000]
train_labels = train_data.target[:1000]

# Test data
test_data = fetch_20newsgroups(
    subset="test",
    categories=["rec.autos", "sci.med"],
    remove=("headers", "footers", "quotes"),
)

test_texts = test_data.data[:500]
test_labels = test_data.target[:500]

print("Training:", len(train_texts))
print("Testing:", len(test_texts))

Training: 1000
Testing: 500


In [8]:
# Simple tokenization and vocabulary
from collections import Counter
import re

def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())[:100]  # Max 100 tokens

# Build vocabulary from training data
all_tokens = [tok for text in train_texts for tok in tokenize(text)]
vocab = {word: idx+2 for idx, (word, _) in enumerate(Counter(all_tokens).most_common(5000))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 5002


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset

In [11]:
# Encode texts
def encode_text(text, max_len=100):
    tokens = tokenize(text)
    encoded = [vocab.get(t, 1) for t in tokens]  # 1 = UNK
    padded = encoded[:max_len] + [0] * (max_len - len(encoded))
    return padded[:max_len]

X_train = torch.tensor([encode_text(t) for t in train_texts])
y_train = torch.tensor(train_labels)
X_test = torch.tensor([encode_text(t) for t in test_texts])
y_test = torch.tensor(test_labels)

print(f"Train shape: {X_train.shape}")

Train shape: torch.Size([1000, 100])


### Exercise B.1: Complete the LSTM Model

In [13]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cpu


In [14]:
# TODO: Complete the LSTM classifier
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout(x)

        # LSTM forward pass
        lstm_out, (hidden, cell) = self.lstm(x)

        # Use the last hidden state
        out = self.fc(hidden[-1])

        return out


# Create model
lstm_model = LSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=64,
    hidden_dim=64,
    num_classes=2
).to(device)

print(f"LSTM Parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

LSTM Parameters: 353,538


In [15]:
# Quick training
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

# Train for 3 epochs
for epoch in range(3):
    lstm_model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = lstm_model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# Evaluate
lstm_model.eval()
with torch.no_grad():
    test_output = lstm_model(X_test.to(device))
    preds = torch.argmax(test_output, dim=1).cpu()
    acc = (preds == y_test).float().mean()
    print(f"\nTest Accuracy: {acc:.4f}")

Epoch 1, Loss: 0.6948
Epoch 2, Loss: 0.6793
Epoch 3, Loss: 0.6652

Test Accuracy: 0.5460


---

# PART C: GRU - News Classification

**Use Case:** Classify news articles by topic.

**Why GRU?** Fewer parameters than LSTM, faster training.

In [17]:
# Load AG News
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

# Use a text classification dataset as a replacement
data = fetch_20newsgroups(
    subset="all",
    categories=[
        "sci.space",
        "rec.sport.baseball",
        "comp.graphics",
        "talk.politics.misc"
    ],
    remove=("headers", "footers", "quotes")
)

texts = data.data
labels = data.target

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

Training samples: 2983
Test samples: 746


### Exercise C.1: Build GRU Classifier

In [19]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

# Four categories to mimic AG News
categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.space",
    "talk.politics.misc"
]

data = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

texts = data.data
labels = data.target

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

# Create ag_train and ag_test in the same format expected by the notebook
ag_train = [
    {"text": text, "label": int(label)}
    for text, label in zip(X_train[:2000], y_train[:2000])
]

ag_test = [
    {"text": text, "label": int(label)}
    for text, label in zip(X_test[:500], y_test[:500])
]

print(f"AG Train: {len(ag_train)}")
print(f"AG Test: {len(ag_test)}")

AG Train: 2000
AG Test: 500


In [20]:
# TODO: Create a GRU classifier (similar to LSTM but using nn.GRU)

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # GRU layer
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)

        # GRU forward pass
        gru_out, hidden = self.gru(x)

        # Use the last hidden state
        out = self.fc(hidden[-1])

        return out


# Build vocabulary and encode (reuse tokenize function)
ag_tokens = [tok for item in ag_train for tok in tokenize(item['text'])]
ag_vocab = {word: idx+2 for idx, (word, _) in enumerate(Counter(ag_tokens).most_common(5000))}
ag_vocab['<PAD>'] = 0
ag_vocab['<UNK>'] = 1

def encode_ag(text, vocab, max_len=50):
    tokens = tokenize(text)
    encoded = [vocab.get(t, 1) for t in tokens]
    return (encoded[:max_len] + [0] * max_len)[:max_len]

X_ag_train = torch.tensor([encode_ag(item['text'], ag_vocab) for item in ag_train])
y_ag_train = torch.tensor([item['label'] for item in ag_train])
X_ag_test = torch.tensor([encode_ag(item['text'], ag_vocab) for item in ag_test])
y_ag_test = torch.tensor([item['label'] for item in ag_test])

print(f"AG News - Train: {X_ag_train.shape}, Test: {X_ag_test.shape}")

AG News - Train: torch.Size([2000, 50]), Test: torch.Size([500, 50])


In [21]:
# Create and train GRU model
gru_model = GRUClassifier(
    vocab_size=len(ag_vocab),
    embed_dim=64,
    hidden_dim=64,
    num_classes=4
).to(device)

print(f"GRU Parameters: {sum(p.numel() for p in gru_model.parameters()):,}")
print(f"(Compare to LSTM: GRU has fewer parameters!)")

GRU Parameters: 345,348
(Compare to LSTM: GRU has fewer parameters!)


---

# PART D: Transformer - Named Entity Recognition

**Use Case:** Extract entities from text.

**Dataset:** CoNLL-2003

In [22]:
# Use pre-trained NER model from Hugging Face
from transformers import pipeline

# Load NER pipeline (uses BERT-based model)
print("Loading NER model...")
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
print("Model loaded!")

Loading NER model...


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded!


In [23]:
# Example NER
text = "Apple Inc. was founded by Steve Jobs in Cupertino, California. Tim Cook is the current CEO."

entities = ner_pipeline(text)
print(f"Text: {text}\n")
print("Entities found:")
for ent in entities:
    print(f"  {ent['word']:20} -> {ent['entity_group']:10} (score: {ent['score']:.3f})")

Text: Apple Inc. was founded by Steve Jobs in Cupertino, California. Tim Cook is the current CEO.

Entities found:
  Apple Inc            -> ORG        (score: 0.999)
  Steve Jobs           -> PER        (score: 0.903)
  Cupertino            -> LOC        (score: 0.998)
  California           -> LOC        (score: 0.999)
  Tim Cook             -> PER        (score: 1.000)


### Exercise D.1: NER on Your Own Texts

In [24]:
# TODO: Write 3 sentences and extract entities
# Include: people, organizations, locations

my_sentences = [
    "Elon Musk visited Paris to attend a technology conference.",
    "Google opened a new research center in London.",
    "Emma Watson studied at the University of Oxford in England.",
]

for sent in my_sentences:
    print(f"\nText: {sent}")
    entities = ner_pipeline(sent)
    for ent in entities:
        print(f"  {ent['word']:20} -> {ent['entity_group']}")


Text: Elon Musk visited Paris to attend a technology conference.
  El                   -> ORG
  ##on Mu              -> PER
  ##sk                 -> ORG
  Paris                -> LOC

Text: Google opened a new research center in London.
  Google               -> ORG
  London               -> LOC

Text: Emma Watson studied at the University of Oxford in England.
  Emma Watson          -> PER
  University of Oxford -> ORG
  England              -> LOC


---

# PART E: LLM - Text Generation with Mistral API

**Use Case:** Conversational AI and Question Answering.

**Setup:** Get a free API key from https://console.mistral.ai/

In [28]:
# TODO: Enter your Mistral API key
# Get free key at: https://console.mistral.ai/

MISTRAL_API_KEY = "YOUR_API_KEY_HERE"

In [27]:
import requests

def query_mistral(prompt, max_tokens=150):
    """Query Mistral API."""
    url = "https://api.mistral.ai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {MISTRAL_API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": "mistral-small-latest",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens
    }

    response = requests.post(url, headers=headers, json=data)
    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content']
    else:
        return f"Error: {response.status_code} - {response.text}"

# Test (only if API key is set)
if MISTRAL_API_KEY:
    response = query_mistral("What is NLP in one sentence?")
    print(f"Mistral: {response}")
else:
    print("Please set your MISTRAL_API_KEY above.")

Mistral: Natural Language Processing (NLP) is a field of AI that enables computers to understand, interpret, and generate human language.


### Exercise E.1: Compare LLM with Traditional Models

In [33]:
# TODO: Ask Mistral to perform sentiment analysis and compare with our LSTM

test_review = "This movie was absolutely terrible. The acting was bad and the plot made no sense."

# LLM approach
if MISTRAL_API_KEY != "7JdD6sySuTWkOtsCtewaPCWfyhia76dJ":
    prompt = f"""Classify the sentiment of this review as 'positive' or 'negative'.
Just respond with one word.

Review: {test_review}

Sentiment:"""

    llm_result = query_mistral(prompt, max_tokens=10)
    print(f"LLM Sentiment: {llm_result}")

# Traditional LSTM approach (if model trained)
try:
    encoded = torch.tensor([encode_text(test_review)]).to(device)
    lstm_model.eval()
    with torch.no_grad():
        lstm_pred = torch.argmax(lstm_model(encoded)).item()
    print(f"LSTM Sentiment: {'positive' if lstm_pred == 1 else 'negative'}")
except:
    print("LSTM model not available")

LSTM Sentiment: negative


In [31]:
# TODO: Use LLM for summarization (something traditional models can't easily do)

long_text = """
Natural language processing (NLP) is a subfield of linguistics, computer science,
and artificial intelligence concerned with the interactions between computers and
human language, in particular how to program computers to process and analyze large
amounts of natural language data. The result is a computer capable of understanding
the contents of documents, including the contextual nuances of the language within them.
"""

prompt = f"""
Summarize the following text in 2-3 sentences.

Text:
{long_text}

Summary:
"""

summary = query_mistral(prompt, max_tokens=100)

print("Summary:")
print(summary)

Summary:
Natural Language Processing (NLP) is a field at the intersection of linguistics, computer science, and AI that focuses on enabling computers to understand, interpret, and analyze human language. It involves processing large volumes of text to extract meaning, including contextual nuances. The goal is to develop systems capable of comprehending documents as humans would.


---

## Final Written Questions (Personal Interpretation)

Answer these questions based on YOUR experiments:

### Question 1: Model Architecture Comparison

Compare the parameter counts you observed:
- RNN: ___ parameters
- LSTM: ___ parameters  
- GRU: ___ parameters

**Why does LSTM have more parameters than GRU?** (Hint: think about gates)

**YOUR ANSWER:**

RNN has the simplest structure and the fewest parameters. It is fast but cannot remember information for a long time. LSTM has the most parameters because it uses memory cells and three gates. It remembers long-term information better but takes more time to train. GRU is between RNN and LSTM. It has fewer parameters than LSTM and trains faster while still giving good performance. Overall, LSTM is usually the most accurate, GRU is a good balance between speed and accuracy, and RNN is the simplest model.

### Question 2: RNN vs LSTM for Long Sequences

**Why would LSTM perform better than vanilla RNN for sentiment analysis on long reviews?** Explain the vanishing gradient problem.

**YOUR ANSWER:**

LSTM performs better because it can remember important information for a longer time. A normal RNN forgets old information when the sequence becomes long because of the vanishing gradient problem. LSTM solves this problem by using memory cells and gates that decide what information to keep and what to forget. This makes LSTM more accurate for long sentences and text classification.

### Question 3: Traditional Models vs LLMs

Based on your experiments:
1. **What can LLMs do that LSTM/GRU cannot?**
2. **What are the disadvantages of using LLM APIs?** (Think: cost, latency, privacy)
3. **When would you choose a traditional model over an LLM?**

**YOUR ANSWER:**

1. LLM advantages:
LLMs understand context better.
They can summarize text, answer questions, translate languages, and generate text.
They work well without training a new model for every task.

2. LLM disadvantages:
They require internet access or API access.
They can be expensive to use.
They are slower than traditional machine learning models.
Sending data to an API may create privacy concerns.

3. When to use traditional models:
Traditional models are better when the task is simple, the dataset is fixed, predictions need to be fast, or privacy is important. They also require fewer computing resources and are easier to deploy

---

## Summary

| Model | Strength | Weakness | Best For |
|-------|----------|----------|----------|
| RNN | Simple, fast | Vanishing gradients | Short sequences |
| LSTM | Long-term memory | More parameters | Long text classification |
| GRU | Efficient, fast | Less expressive | When speed matters |
| Transformer | Parallel, contextual | Expensive | NER, QA, many tasks |
| LLM | Versatile, zero-shot | API cost, latency | Complex reasoning |

---

## Submission

- [ ] All code exercises completed
- [ ] All written questions answered
- [ ] Mistral API tested (or explained why not)
- [ ] **Push to Git and send link to: yoroba93@gmail.com**